# Part 1 — Predictions: Swin, ViT, ResNet50, DenseNet121, EfficientNetB0 + Proposed Ensemble
Collects test-set predictions from 5 single models plus the proposed ensemble
(Swin + ViT + ResNet50, simple average), for the McNemar significance tests.
No `TF_USE_LEGACY_KERAS` needed here. LeViT is done separately in Part 2
(it needs legacy Keras, which conflicts with Swin's `tfswin`).

Run cells top to bottom (or Runtime → Run all).

## Cell 1 — Setup

In [1]:
!pip install -U keras-hub -q
!pip uninstall -y tensorflow-text keras-nlp -q
!pip install tfswin --no-deps -q

import os
import shutil
import zipfile
import json as jsonlib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
import tfswin
from tfswin import SwinTransformerTiny224
import keras_hub
from sklearn.metrics import accuracy_score
import logging
tf.get_logger().setLevel('ERROR')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print("All imports successful!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.31.1 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
All imports successful!


## Cell 2 — Drive mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 — Restore test set

In [3]:
split_zip_path = '/content/drive/MyDrive/split_dataset.zip'
csv_zip_path = '/content/drive/MyDrive/thesis_dataset_csv-20260716T043638Z-1-001.zip'

if not os.path.exists('/content/test'):
    shutil.copy(split_zip_path, '/content/split_dataset.zip')
    with zipfile.ZipFile('/content/split_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')

if not os.path.exists('/content/csv_data'):
    with zipfile.ZipFile(csv_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/csv_data')

print("Test folder ready:", os.path.exists('/content/test'))

Test folder ready: True


## Cell 4 — Load test CSV

In [4]:
test_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/test_data.csv')
test_df['filepath'] = test_df['filepath'].str.replace('/content/split_dataset', '/content')

class_names = sorted(test_df['label'].unique())
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}
true_labels = test_df['label'].map(label_to_index).values

print("Test samples:", len(test_df))
print("Classes:", class_names)

Test samples: 2538
Classes: ['Corn_Common_Rust', 'Corn_Gray_Leaf_Spot', 'Corn_Healthy', 'Corn_Leaf_Blight', 'Rice_Bacterial_Leaf_Blight', 'Rice_Brown_Spot', 'Rice_Healthy', 'Rice_Leaf_Blast', 'Wheat_Brown_Rust', 'Wheat_Healthy', 'Wheat_Loose_Smut', 'Wheat_Yellow_Rust']


## Cell 5 — Config + datasets

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
filepaths = test_df['filepath'].values

def load_uint8(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)

def load_float32(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img

from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input
def load_efficientnet(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return efficientnet_preprocess_input(img)

def make_ds(load_fn):
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds_uint8 = make_ds(load_uint8)              # Swin
test_ds_float32 = make_ds(load_float32)          # ViT, ResNet50, DenseNet121
test_ds_efficientnet = make_ds(load_efficientnet)  # EfficientNetB0
print("Datasets ready")

Datasets ready


## Cell 6 — Load Swin

In [6]:
swin_inputs = layers.Input(shape=(224, 224, 3), dtype='uint8')
swin_base = SwinTransformerTiny224(include_top=False)
x = swin_base(swin_inputs)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
swin_outputs = layers.Dense(num_classes, activation='softmax')(x)
swin_model = models.Model(swin_inputs, swin_outputs)
swin_model.load_weights('/content/drive/MyDrive/thesis_swintiny_outputs/best_swintiny_model.keras')
print("Swin loaded")

177485300/177485300 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Swin loaded


## Cell 7 — Load ViT

In [7]:
vit_backbone = keras_hub.models.ViTBackbone.from_preset("vit_base_patch16_224_imagenet")
vit_backbone.trainable = False
vit_inputs = layers.Input(shape=(224, 224, 3), dtype='float32')
x = layers.Lambda(lambda t: (t / 127.5) - 1.0)(vit_inputs)
x = vit_backbone(x)
cls_token = layers.Lambda(lambda t: t[:, 0, :])(x)
x = layers.Dense(256, activation='relu')(cls_token)
x = layers.Dropout(0.4)(x)
vit_outputs = layers.Dense(num_classes, activation='softmax')(x)
vit_model = models.Model(vit_inputs, vit_outputs)
vit_model.load_weights('/content/drive/MyDrive/thesis_vit_outputs/best_vit_model.keras')
print("ViT loaded")

100%|██████████| 909/909 [00:00<00:00, 822kB/s]


ViT loaded


## Cell 8 — Load ResNet50, DenseNet121, EfficientNetB0

In [8]:
resnet_zip_path = '/content/drive/MyDrive/thesis_resnet50_outputs-20260716T043630Z-1-001.zip'
if not os.path.exists('/content/resnet50_data'):
    with zipfile.ZipFile(resnet_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/resnet50_data')
resnet_model = load_model('/content/resnet50_data/thesis_resnet50_outputs/best_resnet50_model.keras')
print("ResNet50 loaded")

densenet_model = load_model('/content/drive/MyDrive/thesis_densenet121_outputs/best_densenet121_model.keras')
print("DenseNet121 loaded")

efficientnet_model = load_model('/content/drive/MyDrive/best_efficientnet_model.keras')
print("EfficientNetB0 loaded")

ResNet50 loaded
DenseNet121 loaded
EfficientNetB0 loaded


## Cell 9 — Predict with all 5 models + sanity check

In [9]:
pred_swin = swin_model.predict(test_ds_uint8, verbose=1)
pred_vit = vit_model.predict(test_ds_float32, verbose=1)
pred_resnet = resnet_model.predict(test_ds_float32, verbose=1)
pred_densenet = densenet_model.predict(test_ds_float32, verbose=1)
pred_efficientnet = efficientnet_model.predict(test_ds_efficientnet, verbose=1)

for name, preds in [("Swin", pred_swin), ("ViT", pred_vit), ("ResNet50", pred_resnet),
                     ("DenseNet121", pred_densenet), ("EfficientNetB0", pred_efficientnet)]:
    acc = accuracy_score(true_labels, np.argmax(preds, axis=1))
    print(f"{name} test accuracy: {acc:.4f}")

80/80 ━━━━━━━━━━━━━━━━━━━━ 45s 348ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 40s 432ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 20s 176ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 41s 285ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 36s 266ms/step
Swin test accuracy: 0.9842
ViT test accuracy: 0.9823
ResNet50 test accuracy: 0.9795
DenseNet121 test accuracy: 0.9689
EfficientNetB0 test accuracy: 0.9768


## Cell 10 — Proposed ensemble (Swin + ViT + ResNet50, simple average)

In [10]:
ensemble_probs = (pred_swin + pred_vit + pred_resnet) / 3.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)
acc_ensemble = accuracy_score(true_labels, ensemble_preds)
print(f"Proposed ensemble (Swin+ViT+ResNet50) test accuracy: {acc_ensemble:.4f}")

Proposed ensemble (Swin+ViT+ResNet50) test accuracy: 0.9886


## Cell 11 — Save everything to Drive (for the significance-test notebook)

In [11]:
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache'
os.makedirs(output_folder, exist_ok=True)

np.save(f'{output_folder}/pred_swin.npy', pred_swin)
np.save(f'{output_folder}/pred_vit.npy', pred_vit)
np.save(f'{output_folder}/pred_resnet.npy', pred_resnet)
np.save(f'{output_folder}/pred_densenet.npy', pred_densenet)
np.save(f'{output_folder}/pred_efficientnet.npy', pred_efficientnet)
np.save(f'{output_folder}/ensemble_probs.npy', ensemble_probs)
np.save(f'{output_folder}/true_labels.npy', true_labels)
with open(f'{output_folder}/class_names.json', 'w') as f:
    jsonlib.dump(list(class_names), f)

print("Saved all predictions to:", output_folder)
print("Now run Part 2 (LeViT) in a FRESH runtime, then the Significance_Tests notebook.")

Saved all predictions to: /content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache
Now run Part 2 (LeViT) in a FRESH runtime, then the Significance_Tests notebook.


In [12]:
# ============================================================
# Efficiency measurement: parameter count + inference time
# for Swin, ViT, ResNet50, DenseNet121, EfficientNetB0
# Add this as a NEW cell after Cell 11 in SigTest_Part1_Predictions.ipynb
# (reuses the already-loaded models -- no reloading needed)
# ============================================================
import time

def measure_inference_time(model, dataset, warmup=3, timed=15):
    it = iter(dataset.repeat())
    for _ in range(warmup):                      # warm-up (skip tracing/compile overhead)
        batch = next(it)
        _ = model(batch, training=False)
    total_images = 0
    start = time.perf_counter()
    for _ in range(timed):
        batch = next(it)
        _ = model(batch, training=False)
        total_images += batch.shape[0]
    elapsed = time.perf_counter() - start
    ms_per_image = (elapsed / total_images) * 1000
    return ms_per_image

efficiency_rows = []

for name, model, ds in [
    ("Swin", swin_model, test_ds_uint8),
    ("ViT", vit_model, test_ds_float32),
    ("ResNet50", resnet_model, test_ds_float32),
    ("DenseNet121", densenet_model, test_ds_float32),
    ("EfficientNetB0", efficientnet_model, test_ds_efficientnet),
]:
    params = model.count_params()
    ms_per_img = measure_inference_time(model, ds)
    efficiency_rows.append({"Model": name, "Params": params, "ms_per_image": ms_per_img})
    print(f"{name}: {params:,} params, {ms_per_img:.2f} ms/image")

import pandas as pd
efficiency_df = pd.DataFrame(efficiency_rows)
print()
print(efficiency_df.to_string(index=False))

# save to Drive for the final analysis notebook
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache'
efficiency_df.to_csv(f'{output_folder}/efficiency_part1.csv', index=False)
print(f"\nSaved to: {output_folder}/efficiency_part1.csv")

Swin: 27,719,302 params, 21.02 ms/image
ViT: 85,998,604 params, 14.25 ms/image
ResNet50: 24,115,340 params, 11.34 ms/image
DenseNet121: 7,302,988 params, 15.78 ms/image
EfficientNetB0: 4,380,591 params, 10.69 ms/image

         Model   Params  ms_per_image
          Swin 27719302     21.016673
           ViT 85998604     14.246651
      ResNet50 24115340     11.339557
   DenseNet121  7302988     15.775140
EfficientNetB0  4380591     10.690421

Saved to: /content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache/efficiency_part1.csv
